In [19]:
!pip install -r requirements.txt

  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.2
    Uninstalling tenacity-9.1.2:
      Successfully uninstalled tenacity-9.1.2


In [28]:
import dotenv
import os
dotenv.load_dotenv()
key=os.getenv('GEMINI_API_KEY')

In [3]:
from PyPDF2 import PdfReader
def get_pdf_text(pdf_docs):
    text=""
    for pdf in pdf_docs:
        pdf_reader= PdfReader(pdf)
        for page in pdf_reader.pages:
            text+= page.extract_text()
    return  text

ug_rule_text=get_pdf_text(['UG_RULE_BOOK.pdf'])
from langchain.text_splitter import RecursiveCharacterTextSplitter
textSplitter= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks=textSplitter.split_text(ug_rule_text)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain.vectorstores import FAISS
from  google import generativeai as genai
genai.configure(api_key=key)
os.getenv("GOOGLE_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = FAISS.from_texts(chunks, embeddings)
vectorstore.save_local("faiss_index")

Hi!



In [48]:
prompt_template="You are a helpful assistant.Give out as much information as possible, and in detail.Answer using information from the given context only. If the required information is not present in the context, \
    say 'I do not know'.Use simple english. Do not use any asterisks. Do not include any characters apart from letters, numbers \
        and punctuation.  Context: {context} Question: {question}"
from langchain.prompts import PromptTemplate
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)
prompt=PromptTemplate.from_template(prompt_template)
model=ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.2,
)
chain= prompt | model

In [49]:
def respond(question):
    docs=vectorstore.similarity_search(question, k=3)
    context=""
    for doc in docs:
        context+=doc.page_content
    response=chain.invoke({"context":context,"question":question},return_only_outputs=True)
    return response.content

In [50]:
print(respond("Explain academic categories in detail."))

Academic categories are determined based on a student's academic performance over the two most recent regular semesters (Autumn and Spring).  A failing grade of NP is not counted when determining academic standing.

Category I: Students have a Cumulative Performance Index (CPI) of at least 8.0 and no outstanding FR, DX, DR, or W grades in core courses.

Category II: Students have a CPI less than 8.0 and no outstanding FR, DX, DR, or W grades in core courses.

Category III: Students have at least one outstanding FR, DX, DR, or W grade in core courses and at most one FR or DX grade in any other course.  They must have earned at least 18 credits in each of the two preceding semesters.  These 18 credits can be from courses with any TAG.

Category IV: Students have at least one outstanding FR, DX, DR, or W grade in core courses and more than one FR or DX grade in any other course in the two preceding semesters. They must have earned at least 18 credits in each of the two preceding semesters

In [52]:
import streamlit as st
import time
st.title("Interactive IITB UG Rulebook")
st.write("This is a simple RAG application using Langchain and Google Generative AI.")
question = st.text_input("Ask a question about the UG Rulebook:")
while 1:
    if question:
        answer = respond(question)
        st.write("Answer:", answer)
        st.markdown(
            """
            <style>
            .stApp {
                background-image: url('logo.webp');
                background-size: cover;
                background-repeat: no-repeat;
                background-attachment: fixed;
            }
            </style>
            """,
            unsafe_allow_html=True
        )
        question = st.text_input("Ask another question about the UG Rulebook:")
    else:
        st.write("Please enter a question to get started.")
        break
    time.sleep(100)


2025-07-16 02:03:24.980 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-16 02:03:24.981 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-16 02:03:24.982 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-16 02:03:24.982 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-16 02:03:24.982 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-16 02:03:24.983 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-16 02:03:24.983 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-16 02:03:24.984 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar